<a href="https://colab.research.google.com/github/Likelipop/01-NLP/blob/main/Week7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install torch torchvision torchaudio tqdm scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 975.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitl

In [ ]:
!pip install datasets transformers

# Mini-project 1

Apply Transformers & its variants for a text classification problem.

## Problem statement
- Given a set of Transformers variants and a text classification dataset. Your task is to make a complete pipeline from input to output.
- Your pipeline **SHOULD** include the following required components. Each component **SHOULD** be organized into different `class` object.
    - Load & discover dataset
    - Preprocess data
    - Tokenize data
    - Create a DataLoader
    - Build or load model
    - Create a training workflow
    - Set up hyperparameters for training procedure
    - Train model
    - Evaluate model
    - Infer model


In [4]:
# Imports
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments
)
from torch.optim import AdamW

from tqdm import tqdm
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer

In [2]:
# Update the datasets library
!pip install --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 14.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 

In [5]:
import re
from typing import Tuple, Dict, Any
from datasets import load_dataset, DatasetDict, Dataset
from transformers import AutoTokenizer, PreTrainedTokenizer

class MyDataset:
    """
    A dataset handler class that loads, preprocesses, and tokenizes text data for sentiment analysis.

    Attributes:
        dataset_name (str): Name of the dataset to load (e.g., 'imdb').
        model_name (str): Pretrained transformer model name for tokenizer.
        max_length (int): Maximum sequence length for tokenization.
        tokenizer (PreTrainedTokenizer): Hugging Face tokenizer.
        dataset (DatasetDict): Loaded dataset object.
    """

    def __init__(self, dataset_name: str, model_name: str, max_length: int) -> None:
        """
        Initializes the MyDataset object with dataset and tokenizer configurations.

        Args:
            dataset_name (str): Name of the dataset to load.
            model_name (str): Hugging Face model name to load the tokenizer from.
            max_length (int): Max token length for input sequences.
        """
        self.dataset_name = dataset_name
        self.model_name = model_name
        self.max_length = max_length
        self.tokenizer: PreTrainedTokenizer = AutoTokenizer.from_pretrained(model_name)
        self.dataset: DatasetDict | None = None

    def _clean_text(self, example: Dict[str, Any]) -> Dict[str, Any]:
        """
        Cleans the input text by removing punctuation, digits, and keeping only alphabetic words.

        Args:
            example (Dict[str, Any]): A single data sample from the dataset.

        Returns:
            Dict[str, Any]: Cleaned data sample.
        """
        text = example["text"].lower()
        text = re.sub(r"[^\w\s]", "", text)  # Remove punctuation
        text = " ".join(word for word in text.split() if word.isalpha())  # Keep only alphabetic words
        example["text"] = text
        return example

    def _tokenize(self, example: Dict[str, Any]) -> Dict[str, Any]:
        """
        Tokenizes the cleaned text using the specified tokenizer.

        Args:
            example (Dict[str, Any]): A single data sample with cleaned text.

        Returns:
            Dict[str, Any]: Tokenized representation of the sample.
        """
        return self.tokenizer(
            example["text"],
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        )

    def load(self) -> Tuple[Dataset, Dataset]:
        """
        Loads, cleans, tokenizes, and formats the dataset for training and testing.

        Returns:
            Tuple[Dataset, Dataset]: Tuple containing PyTorch-ready training and test datasets.
        """
        self.dataset = load_dataset(self.dataset_name)
        train_dataset = self.dataset["train"]
        test_dataset = self.dataset["test"]

        # Clean text
        train_dataset = train_dataset.map(self._clean_text)
        test_dataset = test_dataset.map(self._clean_text)

        # Tokenize
        train_dataset = train_dataset.map(self._tokenize, batched=True)
        test_dataset = test_dataset.map(self._tokenize, batched=True)

        # Set format for PyTorch
        train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
        test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

        return train_dataset, test_dataset


In [6]:
config = {
    "MODEL_NAME" : "distilbert-base-uncased",
    "MAX_LENGTH" : 256
}

dataset = MyDataset(
    dataset_name = "imdb",
    model_name = config["MODEL_NAME"],
    max_length = config["MAX_LENGTH"]
)

train_dataset, test_dataset = dataset.load()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [7]:
import pprint

pprint.pprint(dataset.dataset["train"])
pprint.pprint(dataset.dataset["train"][0])

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})
{'label': 0,
 'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the '
         'controversy that surrounded it when it was first released in 1967. I '
         'also heard that at first it was seized by U.S. customs if it ever '
         'tried to enter this country, therefore being a fan of films '
         'considered "controversial" I really had to see this for myself.<br '
         '/><br />The plot is centered around a young Swedish drama student '
         'named Lena who wants to learn everything she can about life. In '
         'particular she wants to focus her attentions to making some sort of '
         'documentary on what the average Swede thought about certain '
         'political issues such as the Vietnam War and race issues in the '
         'United States. In between asking politicians and ordinary denizens '
         'of Stockholm about their opinions on politics, she has sex w

In [8]:
pprint.pprint(train_dataset)
pprint.pprint(train_dataset[1])

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 25000
})
{'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
 'input_ids': tensor([  10

In [20]:
import torch
from torch import nn

# 1. Get DataLoaders
import multiprocessing
num_workers = multiprocessing.cpu_count()

In [19]:
class DataLoaderManager:
    def __init__(self, train_dataset, test_dataset, batch_size=16, nw = 1):
        self.train_dataset = train_dataset
        self.test_dataset = test_dataset
        self.batch_size = batch_size
        self.nw = nw

    def get_loaders(self):
        train_loader = DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.nw)
        test_loader = DataLoader(self.test_dataset, batch_size=self.batch_size, num_workers=4)
        return train_loader, test_loader



### BERT
- BERT is a deep learning language model designed to improve the efficiency of natural language processing (NLP) tasks. It is famous for its ability to consider context by analyzing the relationships between words in a sentence bidirectionally. It was introduced by Google researchers in a 2018 paper titled “BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.” Since then, the BERT model has been fine-tuned for use in a variety of fields, including biology, data science, and medicine.

- You could discover [BERT documentation](https://huggingface.co/docs/transformers/en/model_doc/bert) from transformers library @ Huggingface for more details.

- Examples of training use case for Huggingface model: [Huggingface training](https://huggingface.co/docs/transformers/en/training)

In [25]:
class BERTClassifier(nn.Module):
    def __init__(self, model_name: str, dropout: float = 0.3, freeze_bert: bool = True):
        super(BERTClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)

        # Freeze BERT nếu cần
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_output)
        logits = self.classifier(x)
        return logits


In [23]:
class Trainer:
    def __init__(self, model, train_loader, test_loader, lr=2e-5, epochs=3):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.optimizer = AdamW(self.model.parameters(), lr=lr)
        self.epochs = epochs
        self.criterion = nn.CrossEntropyLoss()
        self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=1, gamma=0.95)
        self.scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

    def train(self):
        self.model.train()
        for epoch in range(self.epochs):
            total_loss = 0
            loop = tqdm(self.train_loader, desc=f"Epoch {epoch+1}")

            for batch in loop:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['label'].to(self.device)

                self.optimizer.zero_grad()

                if self.scaler:
                    with torch.cuda.amp.autocast():
                        outputs = self.model(input_ids, attention_mask)
                        loss = self.criterion(outputs, labels)
                    self.scaler.scale(loss).backward()
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    outputs = self.model(input_ids, attention_mask)
                    loss = self.criterion(outputs, labels)
                    loss.backward()
                    self.optimizer.step()

                total_loss += loss.item()
                loop.set_postfix(loss=loss.item())

            avg_loss = total_loss / len(self.train_loader)
            print(f"Epoch {epoch+1} finished. Avg loss: {avg_loss:.4f}")
            torch.cuda.empty_cache()
            self.scheduler.step()

    def evaluate(self):
        self.model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in self.test_loader:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['label'].to(self.device)

                outputs = self.model(input_ids, attention_mask)
                preds = torch.argmax(outputs, dim=1)

                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        acc = accuracy_score(all_labels, all_preds)
        print(f"Test Accuracy: {acc:.4f}")
        return acc


In [ ]:


loader = DataLoaderManager(train_dataset, test_dataset, batch_size=16, nw = num_workers)
train_loader, test_loader = loader.get_loaders()

# 2. Build model
# freeze_bert=True: chỉ train classifier
model = BERTClassifier("bert-base-uncased", freeze_bert=True)


# 3. Train and evaluate
trainer = Trainer(model=model, train_loader=train_loader, test_loader=test_loader,
                  lr=2e-5, epochs=3)

trainer.train()
trainer.evaluate()


Epoch 1:   1%|          | 18/1563 [05:24<7:42:56, 17.98s/it, loss=0.642]

### XLNet

The XLNet model was proposed in XLNet: Generalized Autoregressive Pretraining for Language Understanding by ([Yang et al., 2019](https://arxiv.org/pdf/1906.08237)). XLnet is an extension of the Transformer-XL model pre-trained using an autoregressive method to learn bidirectional contexts by maximizing the expected likelihood over all permutations of the input sequence factorization order.

In [ ]:
from transformers import XLNetTokenizer, XLNetForSequenceClassification

In [ ]:
tokenizer = XLNetTokenizer.from_pretrained('xlnet-base-cased', do_lower_case=True)

tokenized_texts = imdb_train_set[0]['text']
tokenizer(tokenized_texts)

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/760 [00:00<?, ?B/s]

{'input_ids': [17, 150, 14210, 17, 150, 569, 8595, 13, 27886, 40, 94, 894, 1515, 149, 20, 71, 18, 6289, 29, 5197, 36, 90, 36, 30, 89, 528, 25, 5079, 9, 17, 150, 77, 1133, 29, 38, 89, 36, 30, 4665, 37, 17, 660, 9, 23, 9, 9268, 108, 36, 545, 1000, 22, 2137, 52, 234, 19, 2412, 163, 24, 3054, 20, 2701, 957, 17, 12, 1865, 2530, 7257, 2359, 12, 17, 150, 343, 54, 22, 197, 52, 28, 1546, 9, 6461, 2369, 17, 167, 3151, 6461, 2369, 17, 167, 3151, 305, 4487, 27, 12163, 199, 24, 673, 17, 23, 7954, 1406, 3733, 1190, 812, 17, 529, 597, 61, 1578, 22, 1184, 931, 85, 64, 75, 235, 9, 25, 1244, 85, 1578, 22, 1304, 62, 1219, 23, 22, 441, 106, 1968, 20, 6313, 31, 113, 18, 905, 17, 23, 7954, 93, 449, 75, 1028, 413, 705, 148, 34, 18, 17, 9209, 46, 8353, 375, 21, 878, 705, 25, 18, 9114, 1035, 9, 25, 161, 2860, 4380, 21, 5595, 17, 1426, 2252, 3673, 20, 1002, 18242, 75, 58, 7116, 31, 2523, 19, 85, 51, 2216, 33, 62, 3733, 2804, 19, 21700, 19, 21, 1492, 416, 9, 6461, 2369, 17, 167, 3151, 6461, 2369, 17, 167, 3151, 

In [ ]:
model = XLNetForSequenceClassification.from_pretrained("xlnet-base-cased",
                                                       num_labels=1)
model

pytorch_model.bin:   0%|          | 0.00/467M [00:00<?, ?B/s]

Some weights of XLNetForSequenceClassification were not initialized from the model checkpoint at xlnet-base-cased and are newly initialized: ['logits_proj.bias', 'logits_proj.weight', 'sequence_summary.summary.bias', 'sequence_summary.summary.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


XLNetForSequenceClassification(
  (transformer): XLNetModel(
    (word_embedding): Embedding(32000, 768)
    (layer): ModuleList(
      (0-11): 12 x XLNetLayer(
        (rel_attn): XLNetRelativeAttention(
          (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (ff): XLNetFeedForward(
          (layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (layer_1): Linear(in_features=768, out_features=3072, bias=True)
          (layer_2): Linear(in_features=3072, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (activation_function): GELUActivation()
        )
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (sequence_summary): SequenceSummary(
    (summary): Linear(in_features=768, out_features=768, bias=True)
    (activation): Tanh()
    (first_dropout): Identity()
    (last

In [ ]:
# QC
example_tensor = torch.ones(1, 768).type(torch.long)
example_xlnet_output = model(example_tensor)
example_xlnet_output

XLNetForSequenceClassificationOutput(loss=None, logits=tensor([[-0.3567]], grad_fn=<AddmmBackward0>), mems=(tensor([[[ 0.0381,  0.0197,  0.0264,  ..., -0.0002, -0.0350,  0.0263]],

        [[ 0.0381,  0.0197,  0.0264,  ..., -0.0002, -0.0350,  0.0263]],

        [[ 0.0381,  0.0197,  0.0264,  ..., -0.0002, -0.0350,  0.0263]],

        ...,

        [[ 0.0381,  0.0197,  0.0264,  ..., -0.0002, -0.0350,  0.0263]],

        [[ 0.0381,  0.0197,  0.0264,  ..., -0.0002, -0.0350,  0.0263]],

        [[ 0.0381,  0.0197,  0.0264,  ..., -0.0002, -0.0350,  0.0263]]]), tensor([[[ 0.7508,  1.7381,  0.1382,  ..., -0.0210,  0.2592, -0.1314]],

        [[ 0.7508,  1.7381,  0.1382,  ..., -0.0210,  0.2592, -0.1314]],

        [[ 0.7508,  1.7381,  0.1382,  ..., -0.0210,  0.2592, -0.1314]],

        ...,

        [[ 0.7508,  1.7381,  0.1382,  ..., -0.0210,  0.2592, -0.1314]],

        [[ 0.7508,  1.7381,  0.1382,  ..., -0.0210,  0.2592, -0.1314]],

        [[ 0.7508,  1.7381,  0.1382,  ..., -0.0210,  0.2592,